In [1]:
import os 
import pandas as pd 
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_KEY"),
    base_url="https://api.chatanywhere.tech/v1"
)

In [8]:
folder_path = "builder/txtfile"

In [11]:
file_list = [f for f in os.listdir(folder_path)]

In [47]:
title = []
book_content = []
for f in file_list:
    title.append(f.split(".txt")[0])
    with open(os.path.join(folder_path,f),"r",encoding="utf-8") as f:
        content = f.read()
    book_content.append(content)

In [52]:
df = pd.DataFrame(
    [file_list,
    title,
    book_content]
).T

In [53]:
df.columns = ['file_name',"title","content"]

In [24]:
def analysis_mamual_book(title,content):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":"""我会给你一个一个零件的manual book名称和对应内容，
             #要求：
             1.请帮我提炼出来该零件的名字,
             2.以及manual book里面提供的一写solution,solution只需要是概括性的描述不是具体的操作方案，比如：`make sure the coolant tank float operates correctly`,有多个solution就提取多个，以List的形式返回,
             #Output Format:
             {
             "part_name":提取出来的零件名称,
             "solutions":["make sure the coolant tank float operates correctly",...]#提取出来的solution
             }"""
                   },
                  {"role":"user","content":f"这是manual book的title：{title},这是manual book的内容：{content}"}],
        temperature=0.01
    )
    return response.choices[0].message.content

In [30]:
result = []
for ind,row in df.iterrows():
    print(ind)
    answer = analysis_mamual_book(row['title'],row['content'])
    result.append(answer)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42


In [55]:
df['part_name'] = [eval(i)['part_name'] for i in result]
df['solutions'] = [eval(i)['solutions'] for i in result]

In [60]:
df['part_name'].value_counts()

part_name
Turret Indexer Assembly                     3
Haas Oil Skimmer                            3
Lathe                                       2
I/O PCB                                     2
Haas Robot Package                          2
Coolant Refill                              2
Chip Conveyor                               2
Vector Drive                                1
TSC-300/1K                                  1
Through-Tool Air Blast (TAB)                1
Standard Flood Coolant                      1
Proximity Sensor                            1
Spindle Non-Contact Encoder (NCE)           1
Spindle Minimum Lubrication System          1
Lathe Spindle                               1
VMC Side Mount Tool Changer - Double Arm    1
Solenoid                                    1
Sigma 5 - Axis Servo Motor and Cables       1
Servo Amplifier                             1
PSUP PCB                                    1
Live Tooling                                1
Mist Condenser          

In [59]:
df.loc[df['part_name'] == "Haas Robot Package (HRP)","part_name"] = "Haas Robot Package"

In [6]:
part_category_mapping = {
    # 主轴与传动系统（Spindle & Drive）
    "Drive Belt": "Spindle & Drive",
    "Lathe Spindle": "Spindle & Drive",
    "Sigma 5 - Axis Servo Motor and Cables": "Spindle & Drive",
    "Servo Amplifier": "Spindle & Drive",
    "Vector Drive": "Spindle & Drive",
    "Spindle Non-Contact Encoder (NCE)": "Spindle & Drive",
    "BMT65/75 Turret - Live Tool Drive": "Spindle & Drive",  # 合并到 Spindle & Drive
    "VMC Side Mount Tool Changer - Double Arm": "Spindle & Drive",  # 合并到 Spindle & Drive

    # 冷却与润滑系统（Coolant & Lubrication）
    "Coolant Refill": "Coolant & Lubrication",
    "High Pressure Flood Coolant": "Coolant & Lubrication",
    "Standard Flood Coolant": "Coolant & Lubrication",
    "Mist Condenser": "Coolant & Lubrication",
    "Spindle Minimum Lubrication System": "Coolant & Lubrication",
    "Mechanical Bijur Lubrication Pump": "Coolant & Lubrication",
    "Haas Oil Skimmer": "Coolant & Lubrication",
    "Through-Tool Air Blast (TAB)": "Coolant & Lubrication",
    "TSC-300/1K": "Coolant & Lubrication",

    # 电气与控制系统（Electrical & Control）
    "PSUP PCB": "Electrical & Control",
    "I/O PCB": "Electrical & Control",
    "Electrical Safety Door Interlocks": "Electrical & Control",
    "Proximity Sensor": "Electrical & Control",
    "Solenoid": "Electrical & Control",

    # 液压与气动系统（Hydraulic & Pneumatic）
    "Hydraulic Power Unit": "Hydraulic & Pneumatic",
    "Lathe HPU": "Hydraulic & Pneumatic",
    "Hydraulic Tailstock": "Hydraulic & Pneumatic",

    # 机械组件（Mechanical Components）
    "Chip Auger": "Mechanical Components",
    "Chip Conveyor": "Mechanical Components",
    "Way Cover": "Mechanical Components",
    "Foot Pedal": "Mechanical Components",
    "Lathe": "Mechanical Components",  # 合并 Lathe 到此类

    # 自动化与机器人系统（Automation & Robotics）
    "Haas Robot Package": "Automation & Robotics",
    "Parts Catcher": "Automation & Robotics",
    "Turret Indexer Assembly": "Spindle & Drive",
    "Live Tooling": "Spindle & Drive",
}

In [75]:
df['category'] = df['part_name'].map(part_category_mapping)

In [78]:
df[['part_name','category']]

,part_name,category
0,Turret Indexer Assembly,Tooling & Turret
1,BMT65/75 Turret - Live Tool Drive,Spindle & Drive
2,Chip Auger,Mechanical Components
3,Chip Conveyor,Mechanical Components
4,Chip Conveyor,Mechanical Components
5,Coolant Refill,Coolant & Lubrication
6,Coolant Refill,Coolant & Lubrication
7,Drive Belt,Spindle & Drive
8,Electrical Safety Door Interlocks,Electrical & Control
9,Foot Pedal,Mechanical Components


In [9]:
df.to_excel("builder/classified.xlsx",index=False)